In [1]:
import os
import json
from dotenv import load_dotenv
import anthropic
import yfinance as yf
from datetime import datetime

load_dotenv()
client = anthropic.Anthropic()
print("Setup complete")

Setup complete


In [2]:
def get_stock_price(ticker):
    """Fetch the most recent price for a stock ticker."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period="5d")
        if data.empty:
            return f"Error: No data found for {ticker}"
        price = data["Close"].iloc[-1]
        date = data.index[-1].date()
        return f"{ticker} most recent price: ${price:.2f} (as of {date})"
    except Exception as e:
        return f"Error fetching {ticker}: {str(e)}"


def get_historical_return(ticker, years):
    """Compute total return of a stock over N years."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period=f"{years + 1}y")
        if data.empty or len(data) < 2:
            return f"Error: Insufficient data for {ticker}"
        start_price = data["Close"].iloc[0]
        end_price = data["Close"].iloc[-1]
        total_return = ((end_price - start_price) / start_price) * 100
        start_date = data.index[0].date()
        end_date = data.index[-1].date()
        return (f"{ticker} {years}-year return: {total_return:.2f}% "
                f"(${start_price:.2f} to ${end_price:.2f}, "
                f"{start_date} to {end_date})")
    except Exception as e:
        return f"Error computing return for {ticker}: {str(e)}"


def get_fundamentals(ticker):
    """
    Fetch the top 10 fundamental data points for value investors.
    Returns a dict keyed by numbered metric names.
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        def pct(val):
            if val is None:
                return "N/A"
            return f"{val * 100:.1f}%"

        def multiple(val, decimals=1):
            if val is None:
                return "N/A"
            return f"{val:.{decimals}f}x"

        def dollar_b(val):
            if val is None:
                return "N/A"
            return f"${val / 1e9:.1f}B"

        market_cap    = info.get("marketCap")
        enterprise_value = info.get("enterpriseValue")
        ebitda        = info.get("ebitda")
        free_cashflow = info.get("freeCashflow")

        ev_ebitda = None
        if enterprise_value and ebitda and ebitda > 0:
            ev_ebitda = enterprise_value / ebitda

        price_to_fcf = None
        if market_cap and free_cashflow and free_cashflow > 0:
            price_to_fcf = market_cap / free_cashflow

        fcf_yield = None
        if market_cap and free_cashflow and market_cap > 0:
            fcf_yield = free_cashflow / market_cap

        return {
            "company_name":      info.get("longName", ticker),
            "sector":            info.get("sector", "N/A"),
            "industry":          info.get("industry", "N/A"),
            "market_cap":        dollar_b(market_cap),
            "fifty_two_week_high": f"${info.get('fiftyTwoWeekHigh', 0):.2f}",
            "fifty_two_week_low":  f"${info.get('fiftyTwoWeekLow', 0):.2f}",
            # Top 10 — numbered keys for easy table lookup
            "1_trailing_pe":     multiple(info.get("trailingPE")),
            "2_forward_pe":      multiple(info.get("forwardPE")),
            "3_ev_ebitda":       multiple(ev_ebitda),
            "4_price_to_book":   multiple(info.get("priceToBook")),
            "5_price_to_fcf":    multiple(price_to_fcf),
            "6_operating_margin":pct(info.get("operatingMargins")),
            "7_return_on_equity":pct(info.get("returnOnEquity")),
            "8_revenue_growth":  pct(info.get("revenueGrowth")),
            "9_debt_to_equity":  multiple(info.get("debtToEquity"), decimals=2),
            "10_fcf_yield":      pct(fcf_yield),
        }
    except Exception as e:
        return f"Error fetching fundamentals for {ticker}: {str(e)}"


# Quick test
print(json.dumps(get_fundamentals("AVGO"), indent=2))

{
  "company_name": "Broadcom Inc.",
  "sector": "Technology",
  "industry": "Semiconductors",
  "market_cap": "$2001.6B",
  "fifty_two_week_high": "$429.31",
  "fifty_two_week_low": "$184.02",
  "1_trailing_pe": "82.6x",
  "2_forward_pe": "23.3x",
  "3_ev_ebitda": "55.2x",
  "4_price_to_book": "25.1x",
  "5_price_to_fcf": "78.5x",
  "6_operating_margin": "44.9%",
  "7_return_on_equity": "33.4%",
  "8_revenue_growth": "29.5%",
  "9_debt_to_equity": "82.70x",
  "10_fcf_yield": "1.3%"
}


In [3]:
def get_comps_data(comp_tickers):
    """
    Fetch fundamentals for a list of comparable company tickers.
    Returns a dict: {ticker: fundamentals_dict or None}
    """
    comps = {}
    for ticker in comp_tickers:
        ticker = ticker.upper()
        print(f"  Fetching comps: {ticker}...")
        result = get_fundamentals(ticker)
        if isinstance(result, dict):
            comps[ticker] = result
        else:
            print(f"  Warning — could not fetch {ticker}: {result}")
            comps[ticker] = None
    return comps

# Quick test
test_comps = get_comps_data(["AAPL", "NVDA"])
for ticker, data in test_comps.items():
    print(f"{ticker}: {data['company_name'] if data else 'FAILED'}")

  Fetching comps: AAPL...
  Fetching comps: NVDA...
AAPL: Apple Inc.
NVDA: NVIDIA Corporation


In [4]:
data_tools = [
    {
        "name": "get_stock_price",
        "description": "Fetch the most recent closing price for a stock ticker.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Stock ticker symbol"}
            },
            "required": ["ticker"]
        }
    },
    {
        "name": "get_historical_return",
        "description": "Compute total percentage return of a stock over N years.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string"},
                "years": {"type": "integer", "description": "Number of years (1 or 5)"}
            },
            "required": ["ticker", "years"]
        }
    },
    {
        "name": "get_fundamentals",
        "description": (
            "Fetch the top 10 value investor fundamentals: "
            "trailing P/E, forward P/E, EV/EBITDA, price/book, price/FCF, "
            "operating margin, ROE, revenue growth, debt/equity, FCF yield."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string"}
            },
            "required": ["ticker"]
        }
    }
]

print(f"Tools registered: {[t['name'] for t in data_tools]}")

Tools registered: ['get_stock_price', 'get_historical_return', 'get_fundamentals']


In [5]:
def execute_tool(name, inputs):
    if name == "get_stock_price":
        return get_stock_price(**inputs)
    elif name == "get_historical_return":
        return get_historical_return(**inputs)
    elif name == "get_fundamentals":
        return get_fundamentals(**inputs)
    else:
        return f"Error: unknown tool '{name}'"


def gather_market_data(ticker):
    """Run the agent loop to gather all market data for the subject ticker."""
    system_prompt = """You are a quantitative equity research analyst.
    Given a stock ticker, gather comprehensive market data using the available tools.
    Always fetch ALL of the following — do not skip any:
      1. Current price
      2. 1-year historical return
      3. 5-year historical return
      4. Fundamental data (top 10 value investor metrics)
    Call all four tools before responding."""

    messages = [{
        "role": "user",
        "content": (
            f"Gather all market data for {ticker}. "
            f"Fetch current price, 1-year return, 5-year return, and fundamentals."
        )
    }]

    print(f"Gathering market data for {ticker}...")

    while True:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=2048,
            system=system_prompt,
            tools=data_tools,
            messages=messages
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            final_text = next(
                (b.text for b in response.content if b.type == "text"), ""
            )
            return messages, final_text

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  Calling {block.name}({block.input})")
                result = execute_tool(block.name, block.input)
                result_str = (
                    json.dumps(result) if isinstance(result, dict)
                    else str(result)
                )
                print(f"  → {result_str[:100]}...")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result_str
                })

        messages.append({"role": "user", "content": tool_results})

print("Data gathering loop defined")

Data gathering loop defined


In [6]:
research_report_tool = {
    "name": "produce_research_report",
    "description": (
        "Produce a concise structured equity research report. "
        "Use only actual gathered data — never estimate or fabricate figures."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "ticker":            {"type": "string"},
            "company_name":      {"type": "string"},
            "sector":            {"type": "string"},
            "industry":          {"type": "string"},
            "current_price":     {"type": "number"},
            "fifty_two_week_high": {"type": "string"},
            "fifty_two_week_low":  {"type": "string"},
            "one_year_return_pct": {"type": "number"},
            "five_year_return_pct":{"type": "number"},
            "market_cap":        {"type": "string"},
            "trailing_pe":       {"type": "string"},
            "forward_pe":        {"type": "string"},
            "ev_ebitda":         {"type": "string"},
            "price_to_book":     {"type": "string"},
            "price_to_fcf":      {"type": "string"},
            "operating_margin":  {"type": "string"},
            "return_on_equity":  {"type": "string"},
            "revenue_growth_yoy":{"type": "string"},
            "debt_to_equity":    {"type": "string"},
            "fcf_yield":         {"type": "string"},
            "ai_rating": {
                "type": "string",
                "enum": ["Strong Buy", "Buy", "Hold", "Sell", "Strong Sell"],
                "description": "AI rating based solely on gathered data."
            },
            "ai_rating_rationale": {
                "type": "string",
                "description": "One sentence citing specific metrics that drove the rating."
            },
            "investment_thesis": {
                "type": "string",
                "description": "2-3 sentences. Specific and grounded in actual data."
            },
            "key_risks": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Exactly 3 risks. One crisp sentence each."
            },
            "one_line_summary": {
                "type": "string",
                "description": "One punchy sentence under 20 words."
            }
        },
        "required": [
            "ticker", "company_name", "sector", "industry",
            "current_price", "fifty_two_week_high", "fifty_two_week_low",
            "one_year_return_pct", "five_year_return_pct", "market_cap",
            "trailing_pe", "forward_pe", "ev_ebitda",
            "price_to_book", "price_to_fcf",
            "operating_margin", "return_on_equity",
            "revenue_growth_yoy", "debt_to_equity", "fcf_yield",
            "ai_rating", "ai_rating_rationale",
            "investment_thesis", "key_risks", "one_line_summary"
        ]
    }
}

print("Report schema defined")

Report schema defined


In [7]:
def generate_research_report(ticker, conversation_history, data_summary):
    system_prompt = """You are a senior equity research analyst.
    Produce a concise data-driven research report.
    Rules:
    - Use ONLY actual numbers from gathered data. No estimates.
    - AI rating must cite specific metrics.
    - Thesis: 2-3 sentences, specific, data-grounded.
    - Exactly 3 risks, one crisp sentence each.
    - One-line summary under 20 words."""

    messages = conversation_history + [{
        "role": "user",
        "content": (
            f"Based on all market data gathered for {ticker}, "
            f"produce a structured research report.\n\n"
            f"Data summary: {data_summary}"
        )
    }]

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=2048,
        system=system_prompt,
        tools=[research_report_tool],
        tool_choice={"type": "tool", "name": "produce_research_report"},
        messages=messages
    )

    tool_use = next(b for b in response.content if b.type == "tool_use")
    return tool_use.input

print("Report generator defined")

Report generator defined


In [8]:
def format_comps_table(subject_ticker, subject_data, comps_data):
    """
    Build a side-by-side comparison table of all 10 metrics.
    Subject company is in the first data column, clearly marked.
    Comp companies follow to the right.
    """

    # All companies in order: subject first, then comps
    all_tickers = [subject_ticker] + [
        t for t, d in comps_data.items() if d is not None
    ]
    all_data = {subject_ticker: subject_data}
    all_data.update({
        t: d for t, d in comps_data.items() if d is not None
    })

    # Column widths
    LABEL_W = 20        # left-hand metric label column
    COL_W   = 11        # each company column
    n_cols  = len(all_tickers)
    total_w = LABEL_W + (COL_W * n_cols)
    thin    = "  " + "-" * total_w

    # Map: (display label, fundamentals dict key)
    # None key = section header row
    METRICS = [
        ("VALUATION",        None),
        ("1. Trailing P/E",  "1_trailing_pe"),
        ("2. Forward P/E",   "2_forward_pe"),
        ("3. EV / EBITDA",   "3_ev_ebitda"),
        ("4. Price / Book",  "4_price_to_book"),
        ("5. Price / FCF",   "5_price_to_fcf"),
        ("",                 None),   # blank spacer
        ("QUALITY & GROWTH", None),
        ("6. Oper. Margin",  "6_operating_margin"),
        ("7. ROE",           "7_return_on_equity"),
        ("8. Rev. Growth",   "8_revenue_growth"),
        ("9. Debt / Equity", "9_debt_to_equity"),
        ("10. FCF Yield",    "10_fcf_yield"),
    ]

    # Build header row
    # Subject company gets brackets to stand out: [AVGO]
    header = f"  {'METRIC':<{LABEL_W}}"
    for ticker in all_tickers:
        label = f"[{ticker}]" if ticker == subject_ticker else ticker
        header += f"{label:>{COL_W}}"

    # Divider under company names
    company_names_row = f"  {'':<{LABEL_W}}"
    for ticker in all_tickers:
        data = all_data.get(ticker)
        name = data.get("company_name", ticker)[:10] if data else ticker
        company_names_row += f"{name:>{COL_W}}"

    output = f"\n{thin}\n{header}\n{company_names_row}\n{thin}"

    # Build each metric row
    for label, key in METRICS:
        if key is None:
            # Section header or spacer
            if label:
                output += f"\n  {label}"
            else:
                output += "\n"
            continue

        row = f"\n  {label:<{LABEL_W}}"
        for ticker in all_tickers:
            data = all_data.get(ticker)
            val = data.get(key, "N/A") if data else "N/A"
            row += f"{val:>{COL_W}}"
        output += row

    output += f"\n{thin}"
    return output

# Quick test of the formatter with mock data
print("Comps table formatter defined")

Comps table formatter defined


In [9]:
def format_report(r, subject_fundamentals=None, comps_data=None):
    """
    Format the structured report into a clean one-pager.
    Optionally includes a comps table if subject_fundamentals
    and comps_data are provided.
    """
    W   = 60
    div = "=" * W
    thin = "-" * W

    def row(label, value, width=24):
        val = value if value else "N/A"
        if val == "N/A":
            return ""
        return f"\n  {label:<{width}}{val}"

    output = f"""
{div}
  EQUITY RESEARCH — AI GENERATED
{div}
  {r['company_name']} ({r['ticker']})
  {r['sector']} | {r['industry']}
  Generated: {datetime.now().strftime('%B %d, %Y')}
{thin}
  AI RATING: {r['ai_rating'].upper()}
  {r['one_line_summary']}

  Rationale: {r['ai_rating_rationale']}
{div}
  PRICE & PERFORMANCE
{thin}{row('Current Price:', f"${r['current_price']:.2f}")}{row('52-Week Range:', f"{r['fifty_two_week_low']} – {r['fifty_two_week_high']}")}{row('1-Year Return:', f"{r['one_year_return_pct']:.1f}%")}{row('5-Year Return:', f"{r['five_year_return_pct']:.1f}%")}{row('Market Cap:', r['market_cap'])}
{div}
  TOP 10 VALUE INVESTOR FUNDAMENTALS
{thin}
  VALUATION{row('1. Trailing P/E:', r['trailing_pe'])}{row('2. Forward P/E:', r['forward_pe'])}{row('3. EV / EBITDA:', r['ev_ebitda'])}{row('4. Price / Book:', r['price_to_book'])}{row('5. Price / FCF:', r['price_to_fcf'])}

  QUALITY & GROWTH{row('6. Operating Margin:', r['operating_margin'])}{row('7. Return on Equity:', r['return_on_equity'])}{row('8. Revenue Growth:', r['revenue_growth_yoy'])}{row('9. Debt / Equity:', r['debt_to_equity'])}{row('10. FCF Yield:', r['fcf_yield'])}"""

    # Comps table — only shown if comps were provided
    if subject_fundamentals and comps_data:
        valid_comps = {t: d for t, d in comps_data.items() if d is not None}
        if valid_comps:
            output += f"\n{div}\n  COMPARABLE COMPANIES ANALYSIS"
            output += format_comps_table(
                r['ticker'], subject_fundamentals, valid_comps
            )
        else:
            output += f"\n{div}\n  COMPARABLE COMPANIES: No valid comp data retrieved."

    output += f"""
{div}
  INVESTMENT THESIS
{thin}
  {r['investment_thesis']}
{div}
  KEY RISKS
{thin}"""

    for i, risk in enumerate(r['key_risks'], 1):
        output += f"\n  {i}. {risk}"

    output += f"""
{div}
  DISCLAIMER: AI-generated. Not investment advice.
  Verify all data independently before making decisions.
{div}"""

    return output

print("Formatter defined")

Formatter defined


In [10]:
def run_equity_research(ticker, comps=None):
    """
    Full equity research pipeline with optional comps analysis.

    Usage:
        run_equity_research("AVGO")
        run_equity_research("AVGO", comps=["AAPL", "MSFT", "NVDA", "AMD"])
    """
    ticker = ticker.upper()
    comps  = [c.upper() for c in comps] if comps else []

    print(f"\n{'='*60}")
    print(f"  EQUITY RESEARCH: {ticker}")
    if comps:
        print(f"  Comps: {', '.join(comps)}")
    print(f"{'='*60}\n")

    # Step 1: Gather market data via agent loop
    history, summary = gather_market_data(ticker)
    print(f"\n  Market data gathered.")

    # Step 2: Fetch raw fundamentals for subject
    # (needed separately for the comps table)
    print(f"  Fetching subject fundamentals for comps table...")
    subject_fundamentals = get_fundamentals(ticker)
    if not isinstance(subject_fundamentals, dict):
        print(f"  Warning: Could not fetch subject fundamentals.")
        subject_fundamentals = None

    # Step 3: Fetch comps data
    comps_data = {}
    if comps:
        print(f"\n  Fetching comparable companies data...")
        comps_data = get_comps_data(comps)

    # Step 4: Generate structured report via forced tool use
    print(f"\n  Generating structured report...")
    report = generate_research_report(ticker, history, summary)

    # Step 5: Format and display
    formatted = format_report(report, subject_fundamentals, comps_data)
    print(formatted)

    return report, comps_data

print("Pipeline ready.")
print()
print("Usage examples:")
print("  run_equity_research('AVGO')")
print("  run_equity_research('AVGO', comps=['AAPL', 'MSFT', 'NVDA', 'AMD'])")

Pipeline ready.

Usage examples:
  run_equity_research('AVGO')
  run_equity_research('AVGO', comps=['AAPL', 'MSFT', 'NVDA', 'AMD'])


In [11]:
report, comps = run_equity_research(
    "AVGO",
    comps=["NVDA", "INTC", "MRVL", "CSCO"]
)


  EQUITY RESEARCH: AVGO
  Comps: NVDA, INTC, MRVL, CSCO

Gathering market data for AVGO...
  Calling get_stock_price({'ticker': 'AVGO'})
  → AVGO most recent price: $422.76 (as of 2026-04-24)...
  Calling get_historical_return({'ticker': 'AVGO', 'years': 1})
  → AVGO 1-year return: 233.27% ($126.85 to $422.76, 2024-04-25 to 2026-04-24)...
  Calling get_historical_return({'ticker': 'AVGO', 'years': 5})
  → AVGO 5-year return: 1703.70% ($23.44 to $422.76, 2020-04-27 to 2026-04-24)...
  Calling get_fundamentals({'ticker': 'AVGO'})
  → {"company_name": "Broadcom Inc.", "sector": "Technology", "industry": "Semiconductors", "market_cap"...

  Market data gathered.
  Fetching subject fundamentals for comps table...

  Fetching comparable companies data...
  Fetching comps: NVDA...
  Fetching comps: INTC...
  Fetching comps: MRVL...
  Fetching comps: CSCO...

  Generating structured report...

  EQUITY RESEARCH — AI GENERATED
  Broadcom Inc. (AVGO)
  Technology | Semiconductors
  Generated: A